Lead Prediction Model
Objective

Build a supervised machine learning model that predicts whether a customer is likely to convert into an FCY customer.

We'll compare four algorithms:

Logistic Regression (Baseline)
Decision Tree
Random Forest
XGBoost (Optional if installed)

Finally, we'll select the best-performing model.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

import joblib

In [3]:
from xgboost import XGBClassifier

In [4]:
customers = pd.read_csv("customers_feature_engineered.csv")

customers.head()

,CustomerID,Age,Gender,Occupation,MonthlyIncome,Region,Branch,FCYAccount,ETBAccount,MobileBanking,...,HighIncome,FrequentRemittance,HighRemittance,ActiveCustomer,DigitalCustomer,CustomerValueScore,MonthlyRemittanceAverage,FCYUtilizationRate,RelationshipScore,PotentialFCYCustomer
0,100001,59,Female,Business Owner,71321,Addis Ababa,Adama,Yes,Yes,Yes,...,0,0,1,0,0,24368.8,428.75,0.421186,4,0
1,100002,49,Male,Merchant,144028,Amhara,CMC,No,Yes,No,...,1,1,1,0,0,48416.4,701.33,0.211383,6,1
2,100003,35,Female,Teacher,86131,Addis Ababa,Bole,No,Yes,Yes,...,1,0,1,0,0,29201.8,493.75,0.365063,4,0
3,100004,63,Female,Merchant,37963,Tigray,Megenagna,Yes,Yes,No,...,0,0,0,0,0,13896.4,317.92,0.411271,5,0
4,100005,28,Male,Private Employee,113757,Amhara,CMC,No,Yes,No,...,1,0,0,0,0,36243.6,252.75,0.532476,4,0


In [5]:
categorical_columns = [

"Gender",

"Occupation",

"Region",

"Branch",

"FCYAccount",

"ETBAccount",

"MobileBanking",

"InternetBanking",

"AgeGroup"

]

encoder = LabelEncoder()

for col in categorical_columns:

    customers[col] = encoder.fit_transform(customers[col])

In [6]:
#Define Features and Target

#Remove CustomerID because it has no predictive value.

X = customers.drop(
    columns=[
        "CustomerID",
        "LeadStatus"
    ]
)

y = customers["LeadStatus"]

print(X.shape)
print(y.shape)

(500, 27)
(500,)


In [7]:
#Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [8]:
#Scale Numeric Features

numeric_columns = [

"Age",

"MonthlyIncome",

"RemittanceCount",

"TotalRemittanceUSD",

"AverageTransactionUSD",

"FCYPurchaseUSD",

"LastTransactionDays",

"ExistingProducts",

"CustomerValueScore",

"MonthlyRemittanceAverage",

"FCYUtilizationRate",

"RelationshipScore"

]

scaler = StandardScaler()

X_train[numeric_columns] = scaler.fit_transform(
    X_train[numeric_columns]
)

X_test[numeric_columns] = scaler.transform(
    X_test[numeric_columns]
)

In [9]:
#Logistic Regression

lr = LogisticRegression(
    random_state=42,
    max_iter=1000
)

lr.fit(X_train, y_train)

lr_predictions = lr.predict(X_test)

In [10]:
print("Logistic Regression")

print("Accuracy :", accuracy_score(y_test, lr_predictions))

print("Precision:", precision_score(y_test, lr_predictions))

print("Recall   :", recall_score(y_test, lr_predictions))

print("F1 Score :", f1_score(y_test, lr_predictions))

Logistic Regression
Accuracy : 0.99
Precision: 1.0
Recall   : 0.9473684210526315
F1 Score : 0.972972972972973


In [11]:
#Decision Tree

dt = DecisionTreeClassifier(

    random_state=42,

    max_depth=5

)

dt.fit(X_train,y_train)

dt_predictions = dt.predict(X_test)

In [12]:
print("Decision Tree")

print("Accuracy :", accuracy_score(y_test, dt_predictions))

print("Precision:", precision_score(y_test, dt_predictions))

print("Recall   :", recall_score(y_test, dt_predictions))

print("F1 Score :", f1_score(y_test, dt_predictions))

Decision Tree
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


In [13]:
#Random Forest
rf = RandomForestClassifier(

    n_estimators=200,

    random_state=42

)

rf.fit(X_train,y_train)

rf_predictions = rf.predict(X_test)

In [14]:
print("Random Forest")

print("Accuracy :", accuracy_score(y_test, rf_predictions))

print("Precision:", precision_score(y_test, rf_predictions))

print("Recall   :", recall_score(y_test, rf_predictions))

print("F1 Score :", f1_score(y_test, rf_predictions))

Random Forest
Accuracy : 0.99
Precision: 1.0
Recall   : 0.9473684210526315
F1 Score : 0.972972972972973


In [15]:
#XGBoost (Optional)

xgb = XGBClassifier(

    random_state=42,

    n_estimators=200,

    learning_rate=0.1,

    max_depth=5,

    eval_metric="logloss"

)

xgb.fit(X_train,y_train)

xgb_predictions = xgb.predict(X_test)

In [16]:
print("XGBoost")

print("Accuracy :", accuracy_score(y_test, xgb_predictions))

print("Precision:", precision_score(y_test, xgb_predictions))

print("Recall   :", recall_score(y_test, xgb_predictions))

print("F1 Score :", f1_score(y_test, xgb_predictions))

XGBoost
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


In [17]:
#Compare Models
results = pd.DataFrame({

"Model":[

"Logistic Regression",

"Decision Tree",

"Random Forest",

"XGBoost"

],

"Accuracy":[

accuracy_score(y_test,lr_predictions),

accuracy_score(y_test,dt_predictions),

accuracy_score(y_test,rf_predictions),

accuracy_score(y_test,xgb_predictions)

],

"Precision":[

precision_score(y_test,lr_predictions),

precision_score(y_test,dt_predictions),

precision_score(y_test,rf_predictions),

precision_score(y_test,xgb_predictions)

],

"Recall":[

recall_score(y_test,lr_predictions),

recall_score(y_test,dt_predictions),

recall_score(y_test,rf_predictions),

recall_score(y_test,xgb_predictions)

],

"F1 Score":[

f1_score(y_test,lr_predictions),

f1_score(y_test,dt_predictions),

f1_score(y_test,rf_predictions),

f1_score(y_test,xgb_predictions)

]

})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.99,1.0,0.947368,0.972973
1,Decision Tree,1.00,1.0,1.000000,1.000000
2,Random Forest,0.99,1.0,0.947368,0.972973
3,XGBoost,1.00,1.0,1.000000,1.000000


In [18]:
#Select the Best Model
best_model = xgb

In [19]:
#Save the Model

joblib.dump(

best_model,

"lead_prediction_model.pkl"

)

print("Model Saved Successfully")

Model Saved Successfully


In [20]:
#Test Prediction on One Customer

sample_customer = X_test.iloc[[0]]

prediction = best_model.predict(sample_customer)

probability = best_model.predict_proba(sample_customer)

print("Prediction :", prediction)

print("Probability :", probability)

Prediction : [0]
Probability : [[0.99798423 0.00201577]]


In [21]:
#Predict All Customers

customers["PredictedLead"] = best_model.predict(X)

customers["LeadProbability"] = best_model.predict_proba(X)[:,1]

In [22]:
#Rank Customers

ranked_customers = customers.sort_values(

by="LeadProbability",

ascending=False

)

ranked_customers.head(20)

,CustomerID,Age,Gender,Occupation,MonthlyIncome,Region,Branch,FCYAccount,ETBAccount,MobileBanking,...,HighRemittance,ActiveCustomer,DigitalCustomer,CustomerValueScore,MonthlyRemittanceAverage,FCYUtilizationRate,RelationshipScore,PotentialFCYCustomer,PredictedLead,LeadProbability
324,100325,23,0,7,84240,5,7,0,1,1,...,1,0,1,28403.5,455.25,0.305327,5,1,1,0.992390
352,100353,37,0,8,136982,2,5,0,1,1,...,1,0,1,45106.6,602.00,0.262874,5,1,1,0.992239
189,100190,26,0,5,89576,3,6,0,1,1,...,0,0,1,29180.8,318.00,0.352463,5,1,1,0.992239
149,100150,52,1,4,115409,6,0,0,1,1,...,1,0,1,39710.7,681.33,0.273239,8,1,1,0.992141
367,100368,55,1,4,133237,6,2,0,1,1,...,0,0,1,42831.6,376.75,0.231365,6,1,1,0.992141
81,100082,60,0,0,161381,2,2,0,1,1,...,1,0,1,52911.8,682.92,0.271751,5,1,1,0.992026
208,100209,23,0,0,90714,2,1,0,1,1,...,1,0,1,32044.2,771.67,0.307667,4,1,1,0.992026
299,100300,60,1,5,175189,0,2,0,1,1,...,0,0,1,54580.7,304.00,0.295230,4,1,1,0.991985
107,100108,55,1,3,149710,2,4,0,1,1,...,1,0,1,50053.0,723.33,0.252304,7,1,1,0.991765
26,100027,42,1,2,138503,4,4,0,1,1,...,0,0,1,43615.9,277.50,0.339339,5,1,1,0.991765


In [ ]:
#Save Result

ranked_customers.to_csv(

"Lead_Predictions.csv",

index=False

)

print("Lead Predictions Saved")

Important Improvement for a Real Banking Project

There is one issue with our synthetic dataset: LeadStatus was generated using fixed business rules (e.g., remittance count, FCY account status, mobile banking). Because of that, the model may achieve very high accuracy simply by learning those rules.

In a real bank:

LeadStatus would come from historical campaign outcomes (customers who actually accepted or rejected an FCY offer).
The model would learn more subtle patterns and produce more realistic performance (often 75–90% accuracy rather than near-perfect scores).

For learning and portfolio purposes, this project structure is excellent, but for production deployment, the target variable should always be based on real historical conversions, not simulated rules.